In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

# We set-up a simple time-independent two-well system

We start with the usual equation of the resonant flux-biased ATS Hamiltonian in the RWA
\begin{equation}
    \begin{cases}
    \dot a = -i\beta_a'a^*b\\
    \dot b = -i\beta_b'a^2/2-i\varepsilon_d'/2-\kappa_b'b
    \end{cases}
\end{equation}
Then, we perform adiabatic elimination $\dot b = 0$ and we get $$\dot a = -i\beta_a'a^*(i\beta_b'a^2/2\kappa_b'-i\varepsilon_d'/2\kappa_b')$$

We introduce the quadratures $x=(a+a^*)/\sqrt 2$ and $p=i(a^*-a)/\sqrt 2$ 
\begin{equation}
    \begin{cases}
    \dot x = - \aleph (x^2+p^2)x - \beth x\\
    \dot p = - \aleph (x^2+p^2)p + \beth p
    \end{cases},
\end{equation}
with $\aleph = \beta_a'\beta_b'/4\kappa_b'$ and $\beth = \beta_a'\varepsilon_d'/2\kappa_b'$.

Then we can define a potential $U(x,p)$ so that $\dot x = \partial_xU$ and $\dot p = \partial_pU$ :$$U(x,p) = \aleph(x^2+p^2)^2/4+\beth(x^2-p^2)/2$$



In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_a^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2pi*1.5

#Environment
κ_b=1/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.2
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
ε_d_p = ε_d*φ_b/ω_b
κ_b_p = κ_b / ω_b

ℵ = β_a_p*β_b_p/(4*κ_b_p)
ℶ = β_a_p*ε_d_p/(2*κ_b_p)
params = (ℵ, ℶ);

In [ ]:
sqrt(2*ε_d_p/β_b_p)

In [ ]:
function equation!(du,u,params,t)
    ℵ,ℶ = params

    x = u[1]
    p = u[2]
        
    du[1] = -ℵ*(x^2+p^2)*x-ℶ*x
    du[2] = -ℵ*(x^2+p^2)*p+ℶ*p
    return nothing
end

In [ ]:
t_min = 0.0
t_max = 2/κ_b_p 
tspan = (t_min, t_max)
x_0, p_0 = 0.1, -0.01
u_0 = [x_0, p_0]
prob = ODEProblem(equation!, u_0, tspan, params)
sol = solve(prob, Tsit5())
t = sol.t
x = [u[1] for u in sol.u]
p = [u[2] for u in sol.u];

In [ ]:
x_all = -.2:0.01:.2
p_all = -.35:0.01:.35

U = [ℵ*(x^2 + p^2)^2/4 + ℶ*(x^2 - p^2)/2 for p in p_all, x in x_all]

heatmap(x_all, p_all, U)
scatter!(x,p, label="Trajectory")

# Writing the Hamiltonian and initializing the GMAM

We now add noise to the system
\begin{equation}
    \begin{cases}
    \dot x = - \aleph (x^2+p^2)x - \beth x + \sqrt \epsilon\eta_x(t)\\
    \dot p = - \aleph (x^2+p^2)p + \beth p + \sqrt \epsilon\eta_p(t)
    \end{cases}.
\end{equation}

The noise is isotropic. We can write MSRDJ action: $$iS=-\int dt [P_x\dot x+P_p\dot p - H(x,p,P_x,P_p)],$$
with the Hamiltonian $$H(x,p,P_x,P_p) = -[\aleph(x^2+p^2)+\beth]xP_x - [\aleph(x^2+p^2)-\beth]pP_p + \epsilon(P_x^2+P_p^2)$$

Inputs: $M = 100 \in \mathbb N$, two stable fixed points $\phi_a = (0, -\sqrt{2\varepsilon_d'/\beta_b'})$ and $\phi_b = (0, 0)$ of the noiseless dynamics where $H(\phi_{a,b},0)=\nabla_{\bm \theta}H(\phi_{a,b},0)=0$; a path $\{\hat\phi_i^0\}_{i\in I}$ with $\hat\phi_0^0 =\phi_a$ and $\hat\phi_M^0 =\phi_b$, such that $|\hat\phi_{i+1}^0-\hat\phi_{i}^0|$ is constant in $i$ (we will take a parabola linking $\phi_a$ and $\phi_b$); the functions $f(\mathbf u, \mathbf v)= \alpha \nabla_{\bm \phi}H - \nabla_{\bm \theta}H$, $g(\mathbf u, \mathbf v)=\alpha \nabla_{\bm \phi}H + \nabla_{\bm \theta}H$ and $\lambda(\mathbf u, \mathbf v) = \biggl[|\langle \nabla_{\bm \theta}H, \partial_s \hat\phi \rangle|+\sqrt{\max(0,\langle \nabla_{\bm \theta}H, \partial_s \hat\phi \rangle^2-4H|\partial_s\hat\phi|^2)}\biggr]/(2|\partial_s\hat\phi|^2)$; $\Delta\tau = 0.1 > 0, \alpha = 1 >0$

We have $\nabla_{\bm\phi}H = \begin{pmatrix} - \aleph(3x^2P_x+p^2P_x+2xpP_p)-\beth P_x \\- \aleph(3p^2P_p+x^2P_p+2pxP_x)+\beth P_p \end{pmatrix}$ and $\nabla_{\bm\theta}H = \begin{pmatrix}-\aleph (x^2+p^2)x-\beth x +2 \epsilon P_x\\ -\aleph (x^2+p^2)p+\beth p+2 \epsilon P_p\end{pmatrix}$

In [ ]:
M = 500
ϕ_a = [0.0, -sqrt(2*ε_d_p/β_b_p)]
ϕ_b = [0.0, 0.0]
α = 1
params = (ℵ,ℶ,1)


# Your parabola: x as function of p (endpoints give x=0 automatically)
#initial_path_relation(p; c=2.0) = c * p * (p - ϕ_a[2])  # x(p)
function initial_path_relation(p; A=0.3, h=0.15)
    p_a = ϕ_a[2]
    t = (p - p_a)/(-p_a)  # 0..1
    return (-p_a) * (A*sin(pi*t) + h*sin(2pi*t))
end

function initial_guess_path(M, ϕ_a, ϕ_b; n_fine=50_000)
    p_a = ϕ_a[2]
    p_b = ϕ_b[2]  # should be 0.0 here

    # 1) Fine sampling in parameter p
    p_f = collect(range(p_a, p_b, length=n_fine))
    x_f = similar(p_f)
    @inbounds for k in eachindex(p_f)
        x_f[k] = initial_path_relation(p_f[k])   # <-- use your new relation here
    end

    # 2) Cumulative arc-length along the polyline
    s = zeros(Float64, n_fine)
    @inbounds for k in 2:n_fine
        dx = x_f[k] - x_f[k-1]
        dp = p_f[k] - p_f[k-1]
        s[k] = s[k-1] + sqrt(dx*dx + dp*dp)
    end
    L = s[end]

    # 3) Target equally spaced arc-lengths
    s_target = collect(range(0.0, L, length=M+1))

    # 4) Interpolate (x,p) at those arc-lengths
    x = Vector{Float64}(undef, M+1)
    p = Vector{Float64}(undef, M+1)

    @inbounds for i in 1:(M+1)
        st = s_target[i]
        k = searchsortedfirst(s, st)

        if k <= 1
            x[i] = x_f[1];  p[i] = p_f[1]
        elseif k > n_fine
            x[i] = x_f[end]; p[i] = p_f[end]
        else
            s0, s1 = s[k-1], s[k]
            t = (st - s0) / (s1 - s0)  # in [0,1]
            x[i] = (1-t)*x_f[k-1] + t*x_f[k]
            p[i] = (1-t)*p_f[k-1] + t*p_f[k]
        end
    end

    # Pack as a 2×(M+1) matrix: row1=x, row2=p
    path = permutedims(hcat(x, p))  # (M+1)×2
    path = permutedims(path)        # 2×(M+1)

    # Enforce exact endpoints (good practice)
    path[1, :] .= ϕ_a
    path[end, :] .= ϕ_b

    return path
end

ϕ_init = initial_guess_path(M, ϕ_a, ϕ_b)

# Quick check: step lengths should be (nearly) constant
gaps = [norm(ϕ_init[i+1,:] - ϕ_init[i,:]) for i in 1:M]
println("gap min/max = ", minimum(gaps), " / ", maximum(gaps))
println("relative spread = ", (maximum(gaps)-minimum(gaps))/(sum(gaps)/length(gaps)))

In [ ]:
function grad_phi_H(x,p,P_x,P_p,params)
    ℵ,ℶ,_ = params
    grad = [-ℵ*(3x^2*P_x+p^2*P_x+2*x*p*P_p)-ℶ*P_x,
            -ℵ*(3p^2*P_p+x^2*P_p+2*p*x*P_x)+ℶ*P_p]
    return(grad)
end

function grad_theta_H(x,p,P_x,P_p,params)
    ℵ,ℶ,ϵ = params
    grad = [-ℵ*(x^2+p^2)x-ℶ*x+2*P_x,
            -ℵ*(x^2+p^2)p+ℶ*p+2*P_p,]
    return(grad)
end

function f(u,v,params,α)
    x, p = (u+v)/2
    P_x, P_p = (u-v)/2/α
    res = α*grad_phi_H(x,p,P_x,P_p,params) - grad_theta_H(x,p,P_x,P_p,params)
    return res
end

function g(u,v,params,α)
    x, p = (u+v)/2
    P_x, P_p = (u-v)/2/α
    res = α*grad_phi_H(x,p,P_x,P_p,params) + grad_theta_H(x,p,P_x,P_p,params)
    return res
end

function H(x,p,P_x,P_p,params)
    ℵ, ℶ, ϵ = params
    return P_x*(-ℵ*(x^2+p^2)*x - ℶ*x) +
           P_p*(-ℵ*(x^2+p^2)*p + ℶ*p) +
           (P_x^2 + P_p^2)
end

function λ_step(u::AbstractVector, v::AbstractVector, dsϕ::AbstractVector, params, α)
    x, p = (u .+ v) ./ 2
    P_x, P_p = (u .- v) ./ (2α)

    gradθ = grad_theta_H(x, p, P_x, P_p, params)
    Hval  = H(x, p, P_x, P_p, params)

    dot_term = dot(gradθ, dsϕ)
    norm2    = dot(dsϕ, dsϕ)

    disc = dot_term^2 - 4*Hval*norm2
    return (abs(dot_term) + sqrt(max(0.0, disc))) / (2*norm2)
end

function λ(u_all::AbstractMatrix, v_all::AbstractMatrix, params, α)
    Mplus1 = size(u_all, 1)
    M = Mplus1 - 1

    ϕ_all = (u_all .+ v_all) ./ 2
    λ_vals = zeros(Float64, M)              # λ_i for i=0..M-1 (Julia 1..M)

    @inbounds for i in 1:M
        dsϕ = view(ϕ_all, i+1, :) .- view(ϕ_all, i, :)
        λ_vals[i] = λ_step(view(u_all,i,:), view(v_all,i,:), dsϕ, params, α)
    end
    return λ_vals
end

Let's intialize the GMAM

In [ ]:
θ_init = zeros((M+1,2))
u_init = ϕ_init + α* θ_init
v_init = ϕ_init - α* θ_init
Δs = 1/M

# Let's perform the gradient descent ascent

We got that for $n\geq0$, we solve for each $i=M,M-1,\dots,1,0$ for the value of $u_i^{n+1}$ through $$u_M^{n+1} = -v_M^n+2\phi_b,$$ $$ u_i^{n+1} = \biggl[\lambda_i(u^n,v^n)u_{i+1}^{n+1}/\Delta s + f(u_i^n,v_i^n)\biggr]/\biggl[1/\Delta\tau + \lambda_i(u^n,v^n)/\Delta s\biggr] , \qquad i=M-1,\dots,1,0$$

For the value of $v_i^{n+1}$ through $$v_0^{n+1} = -u_0^{n+1}+2\phi_a,$$ $$ v_{i}^{n+1} = \biggl[\lambda_i(u^{n+1},v^n)v_{i-1}^{n+1}/\Delta s + g(u_i^{n+1},v_i^n)\biggr]/\biggl[1/\Delta\tau + \lambda_i(u^{n+1},v^n)/\Delta s\biggr] , \qquad i=1,\dots,M-1,M$$

In [ ]:
function update_u(u, v, params, α, ϕ_b, Δτ, Δs)
    Mplus1 = size(u, 1)
    M = Mplus1 - 1

    λ_vals = λ(u, v, params, α)

    u_new = zeros(Float64, Mplus1, 2)

    # u_M^{n+1} = -v_M^n + 2 ϕ_b
    u_new[Mplus1, :] .=  -v[Mplus1,:] + 2*ϕ_b

    # backward recursion: i = M-1,...,0  (Julia: rows M,...,1)
    @inbounds for i in M:-1:1
        num = (λ_vals[i] .* view(u_new, i+1, :) ./ Δs) .+ f(view(u,i,:), view(v,i,:), params, α)
        den = (1/Δτ) + (λ_vals[i] / Δs)
        u_new[i, :] .= num ./ den
    end
    #u_new[1, :] .=  ϕ_a

    return u_new, λ_vals
end

function update_v(u, v, params, α, ϕ_a, Δτ, Δs)
    Mplus1 = size(v, 1)
    M = Mplus1 - 1

    # λ_i(u^{n+1}, v^n): defined on segments i=0..M-1 (Julia 1..M)
    λ_vals = λ(u, v, params, α)

    v_new = zeros(Float64, Mplus1, 2)

    # v_0^{n+1} = -u_0^n + 2 ϕ_a   (row 1)
    v_new[1, :] .= -u[1,:]+2*ϕ_a

    # forward recursion: i = 1..M  (Julia: rows 2..M+1)
    @inbounds for i in 2:(M+1)
        num = (λ_vals[i-1] .* view(v_new, i-1, :) ./ Δs) .+ g(view(u, i, :), view(v, i, :), params, α)
        den = (1/Δτ) + (λ_vals[i-1] / Δs)
        v_new[i, :] .= num ./ den
    end

    #v_new[Mplus1, :] .= ϕ_b

    return v_new, λ_vals
end

using LinearAlgebra


# Linear interpolation helper between rows
@inline function lerp_row(A::AbstractMatrix, k0::Int, k1::Int, t::Float64)
    return (1-t) .* view(A, k0, :) .+ t .* view(A, k1, :)
end

function reparametrize_string!(u_new::AbstractMatrix, v_new::AbstractMatrix, α;
                               n_fine::Int=10_000)
    Mplus1 = size(u_new, 1)
    M = Mplus1 - 1

    # Build φ and θ from u,v
    ϕ = (u_new .+ v_new) ./ 2
    θ = (u_new .- v_new) ./ (2α)

    # Arc-length along φ (polyline)
    s = zeros(Float64, Mplus1)
    @inbounds for i in 2:Mplus1
        d = view(ϕ, i, :) .- view(ϕ, i-1, :)
        s[i] = s[i-1] + norm(d)
    end
    L = s[end]

    # Targets: equally spaced in arc-length
    s_target = collect(range(0.0, L, length=Mplus1))

    # Interpolate φ and θ onto s_target
    ϕr = similar(ϕ)
    θr = similar(θ)

    @inbounds for i in 1:Mplus1
        st = s_target[i]
        k = searchsortedfirst(s, st)

        if k <= 1
            ϕr[i, :] .= view(ϕ, 1, :)
            θr[i, :] .= view(θ, 1, :)
        elseif k > Mplus1
            ϕr[i, :] .= view(ϕ, Mplus1, :)
            θr[i, :] .= view(θ, Mplus1, :)
        else
            s0, s1 = s[k-1], s[k]
            t = (st - s0) / (s1 - s0)

            ϕr[i, :] .= lerp_row(ϕ, k-1, k, t)
            θr[i, :] .= lerp_row(θ, k-1, k, t)
        end
    end

    # Enforce exact endpoints (recommended)
    ϕr[1, :]   .= ϕ[1, :]
    ϕr[end, :] .= ϕ[end, :]
    θr[1, :]   .= θ[1, :]
    θr[end, :] .= θ[end, :]

    # Rebuild u,v from reparametrized φ,θ
    u_new .= ϕr .+ α .* θr
    v_new .= ϕr .- α .* θr

    return nothing
end

Δτ = 1e-3
u=u_init
v=v_init
N_iter = 10
for n in 1:N_iter
    u_new, _ = update_u(u, v, params, α, ϕ_b, Δτ, Δs)          # backward
    v_new, _ = update_v(u_new, v, params, α, ϕ_a, Δτ, Δs)   # forward

    reparametrize_string!(u_new, v_new, α)
    #ϕ_new = (u_new .+ v_new) ./ 2
    #gaps = [norm(view(ϕ_new,i+1,:) .- view(ϕ_new,i,:)) for i in 1:M]

    u, v = u_new, v_new
end


In [ ]:
ϕ_fin = (u+v)/2
plot(ϕ_fin[:, 1], ϕ_fin[:, 2])
xlims!(-.3,.3)
ylims!(-.3,.3)

In [ ]:
plot(ϕ_fin[:, 1])
plot!(ϕ_fin[:, 2])

In [ ]:
θ_fin = (u-v)/2/α
plot(θ_fin[:, 1])
plot!(θ_fin[:, 2])

In [ ]:
phi_all = -.3:0.001:.1
theta_all = -.02:0.001:.02

H_val = [H(0, p, 0, P_p, params) for p in phi_all, P_p in theta_all]


heatmap(theta_all,phi_all, H_val)
xlabel!(L"$P_p$")
ylabel!(L"$p$")
#scatter!(x,p, label="Trajectory")
plot!(θ_fin[:, 2],ϕ_fin[:, 2])

In [ ]:
H_val = zeros(Float64,M+1)
for point in 1:M+1
    x, p = ϕ_fin[point, :]
    P_x, P_p = θ_fin[point, :]
    H_val[point] = H(x,p,P_x,P_p,params)
end
plot(H_val)

In [ ]:
λ_fin = λ(u,v,params,α)
plot(λ_fin)

# Let's focus on the two-mode system with RWA

We start with the usual equation of the resonant flux-biased ATS Hamiltonian in the RWA
\begin{equation}
    \begin{cases}
    \dot a = -i\beta_a'a^*b\\
    \dot b = -i\beta_b'a^2/2-i\varepsilon_d'/2-\kappa_b'b
    \end{cases}
\end{equation}
and we write the equation on the field quadratures
\begin{equation}
    \begin{cases}
    \dot x_a = \beta_a'(x_ap_b-p_ax_b)/\sqrt 2\\
    \dot p_a = -\beta_a'(x_ax_b+p_ap_b)/\sqrt 2\\
    \dot x_b = \beta_b'x_ap_a/\sqrt 2 - \kappa_b'x_b\\
    \dot x_b = -\beta_b'(x_a^2-p_a^2)/2\sqrt 2  - \varepsilon_d'/\sqrt 2- \kappa_b'p_b\\
    \end{cases}
\end{equation}

In [ ]:
function equation_2modes!(du,u,params,t)
    β_a_p,β_b_p,ε_d_p,κ_b_p = params

    x_a = u[1]
    p_a = u[2]
    x_b = u[3]
    p_b = u[4]
        
    du[1] = β_a_p*(x_a*p_b-p_a*x_b)/sqrt(2)
    du[2] = -β_a_p*(x_a*x_b+p_a*p_b)/sqrt(2)
    du[3] = β_b_p*x_a*p_a/sqrt(2)-κ_b_p*x_b
    du[4] = -β_b_p*(x_a^2-p_a^2)/2/sqrt(2)-ε_d_p/sqrt(2)-κ_b_p*p_b
    return nothing
end

In [ ]:
t_min = 0.0
t_max = 10/κ_b_p 
tspan = (t_min, t_max)

a_0 = 0.1-0.1im #1im*sqrt(ε_d_p/β_b_p)
b_0 = -1im*ε_d_p/2/κ_b_p#0

x_a0, p_a0, x_b0, p_b0 = real(a_0)*sqrt(2), imag(a_0)*sqrt(2), real(b_0)*sqrt(2), imag(b_0)*sqrt(2)
u_0 = [x_a0, p_a0, x_b0, p_b0]
params = (β_a_p,β_b_p,ε_d_p,κ_b_p)
prob = ODEProblem(equation_2modes!, u_0, tspan, params)
sol = solve(prob, Tsit5())
t = sol.t
x_a = [u[1] for u in sol.u]
p_a = [u[2] for u in sol.u]
x_b = [u[3] for u in sol.u]
p_b = [u[4] for u in sol.u];

In [ ]:
x_all = -.2:0.01:.2
p_all = -.35:0.01:.35

U = [ℵ*(x^2 + p^2)^2/4 + ℶ*(x^2 - p^2)/2 for p in p_all, x in x_all]

heatmap(x_all, p_all, U)
#plot!(x_a,p_a, label="Trajectory")

In [ ]:
plot(x_b,p_b, label="Trajectory")

We now add noise to the system
\begin{equation}
    \begin{cases}
    \dot x_a = \beta_a'(x_ap_b-p_ax_b)/\sqrt 2 \\
    \dot p_a = -\beta_a'(x_ax_b+p_ap_b)/\sqrt 2\\
    \dot x_b = \beta_b'x_ap_a/\sqrt 2 - \kappa_b'x_b + \sqrt \epsilon \eta_x(t)\\
    \dot p_b = -\beta_b'(x_a^2-p_a^2)/2\sqrt 2  - \varepsilon_d'/\sqrt 2- \kappa_b'p_b  + \sqrt \epsilon \eta_p(t)\\
    \end{cases}
\end{equation}

The noise is isotropic and we got $$\langle\eta_i(t')\eta_j(t)\rangle = 2\delta_{i,j}\delta(t'-t).$$ We can write MSRDJ action: $$iS=-\int dt [P_{x_a}\dot x_a+P_{p_a}\dot p_a+P_{x_b}\dot x_b+P_{p_b}\dot p_b  - H(x_a,p_a,x_b,p_b,P_{x_a},P_{p_a},P_{x_b},P_{p_b})],$$
with the Hamiltonian 
\begin{equation}
\begin{split}
H(x,p,P_x,P_p) =& [\beta_a'(x_ap_b-p_ax_b)/\sqrt 2]P_{x_a} + [-\beta_a'(x_ax_b+p_ap_b)/\sqrt 2]P_{p_a}\\& + [\beta_b'x_ap_a/\sqrt 2 - \kappa_b'x_b]P_{x_b} + [-\beta_b'(x_a^2-p_a^2)/2\sqrt 2  - \varepsilon_d'/\sqrt 2- \kappa_b'p_b]P_{p_b}\\&+\epsilon(P_{x_b}^2+P_{p_b}^2)
\end{split}
\end{equation}

We have $\nabla_{\bm\phi}H = \begin{pmatrix} \beta_a'(p_bP_{x_a}-x_bP_{p_a})/\sqrt 2 + \beta_b'(p_aP_{x_b}-x_aP_{p_b})/\sqrt 2 \\-\beta_a'(x_bP_{x_a}+p_bP_{p_a})/\sqrt 2+\beta_b'(x_aP_{x_b}+p_aP_{p_b})/\sqrt 2\\ -\beta_a'(p_aP_{x_a}+x_aP_{p_a})/\sqrt 2 -\kappa_b'P_{x_b}\\\beta_a'(x_aP_{x_a}-p_aP_{p_a})/\sqrt 2 - \kappa_b'P_{p_b}\end{pmatrix}$ 
and $\nabla_{\bm\theta}H = \begin{pmatrix}\beta_a'(x_ap_b-p_ax_b)/\sqrt 2\\ -\beta_a'(x_ax_b+p_ap_b)/\sqrt 2\\ \beta_b'x_ap_a/\sqrt 2 - \kappa_b'x_b + 2\epsilon P_{x_b}\\-\beta_b'(x_a^2-p_a^2)/2\sqrt 2  - \varepsilon_d'/\sqrt 2- \kappa_b'p_b + 2\epsilon P_{p_b}\end{pmatrix}$

In [ ]:
M = 500
ϕ_a = [0.0, -sqrt(2*ε_d_p/β_b_p), 0.0 ,0.0]
ϕ_b = [0.0, 0.0, 0.0, -sqrt(0.5)*ε_d_p/κ_b_p]
α = 1
ϵ = 0#1e-4
params = (β_a_p,β_b_p,ε_d_p,κ_b_p,ϵ)

# Bump for x_a as a function of p_a (vanishes at endpoints p_a=p_a* and p_a=0)
function initial_path_relation_xa(p_a; A=0.3, h=0.15, p_a0=nothing)
    # p_a0 is the fixed-point value (negative). If not given, infer from ϕ_a later by closure.
    @assert p_a0 !== nothing "Pass p_a0 (the starting p_a) to initial_path_relation_xa"
    t = (p_a - p_a0)/(-p_a0)  # maps p_a0 -> 0 and 0 -> 1
    return (-p_a0) * (A*sin(pi*t) + h*sin(2pi*t))
end
function initial_path_relation_xb(p_b; A=0.3, h=0.15, p_b1=nothing)
    # p_b1 is the target fixed-point value at φ_b (typically negative).
    @assert p_b1 !== nothing "Pass p_b1 (the ending p_b) to initial_path_relation_xb"
    t = p_b / p_b1              # maps 0 -> 0 and p_b1 -> 1
    return (-p_b1) * (A*sin(pi*t) + h*sin(2pi*t))
end

function initial_guess_path_4D(M, ϕ_a, ϕ_b; n_fine=50_000, A=0.3, h=0.15)
    # ϕ = (x_a, p_a, x_b, p_b)
    p_a0 = ϕ_a[2]
    p_b0 = ϕ_a[4]
    p_a1 = ϕ_b[2]
    p_b1 = ϕ_b[4]

    # 1) Fine sampling in a dummy parameter t in [0,1]
    t_f = collect(range(0.0, 1.0, length=n_fine))

    # Build fine path points in 4D
    ϕ_f = zeros(Float64, n_fine, 4)
    @inbounds for k in eachindex(t_f)
        t = t_f[k]

        # linear schedules for momenta
        p_a = (1-t)*p_a0 + t*p_a1
        p_b = (1-t)*p_b0 + t*p_b1

        # x_a bump tied to p_a; x_b kept 0 (can add a bump similarly if desired)
        x_a = initial_path_relation_xa(p_a; A=A, h=h, p_a0=p_a0)
        x_b = initial_path_relation_xb(p_b; A=A, h=h, p_b1=p_b1)

        ϕ_f[k, 1] = x_a
        ϕ_f[k, 2] = p_a
        ϕ_f[k, 3] = x_b
        ϕ_f[k, 4] = p_b
    end

    # 2) Cumulative arc-length in 4D along the polyline
    s = zeros(Float64, n_fine)
    @inbounds for k in 2:n_fine
        d = view(ϕ_f, k, :) .- view(ϕ_f, k-1, :)
        s[k] = s[k-1] + norm(d)
    end
    L = s[end]

    # 3) Target equally spaced arc-lengths
    s_target = collect(range(0.0, L, length=M+1))

    # 4) Interpolate onto equally spaced arc-lengths
    path = zeros(Float64, M+1, 4)

    @inbounds for i in 1:(M+1)
        st = s_target[i]
        k = searchsortedfirst(s, st)

        if k <= 1
            path[i, :] .= view(ϕ_f, 1, :)
        elseif k > n_fine
            path[i, :] .= view(ϕ_f, n_fine, :)
        else
            s0, s1 = s[k-1], s[k]
            τ = (st - s0) / (s1 - s0)  # in [0,1]
            path[i, :] .= (1-τ) .* view(ϕ_f, k-1, :) .+ τ .* view(ϕ_f, k, :)
        end
    end

    # Enforce exact endpoints
    path[1, :]   .= ϕ_a
    path[end, :] .= ϕ_b

    return path
end


ϕ_init = initial_guess_path_4D(M, ϕ_a, ϕ_b)

# Check constant spacing in 4D
gaps = [norm(view(ϕ_init,i+1,:) .- view(ϕ_init,i,:)) for i in 1:M]
println("gap min/max = ", minimum(gaps), " / ", maximum(gaps))
println("relative spread = ", (maximum(gaps)-minimum(gaps)) / (sum(gaps)/length(gaps)))

In [ ]:
function grad_phi_H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params)
    β_a_p,β_b_p,ε_d_p,κ_b_p,_ = params
    grad = [β_a_p*(p_b*P_x_a - x_b*P_p_a)/sqrt(2)+β_b_p*(p_a*P_x_b-x_a*P_p_b)/sqrt(2),
            -β_a_p*(x_b*P_x_a + p_b*P_p_a)/sqrt(2)+β_b_p*(x_a*P_x_b+p_a*P_p_b)/sqrt(2),
            -β_a_p*(p_a*P_x_a+x_a*P_p_a)/sqrt(2)-κ_b_p*P_x_b,
            β_a_p*(x_a*P_x_a-p_a*P_p_a)/sqrt(2)-κ_b_p*P_p_b]
    return(grad)
end

function grad_theta_H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params)
    β_a_p,β_b_p,ε_d_p,κ_b_p,_ = params
    grad = [β_a_p*(x_a*p_b-p_a*x_b)/sqrt(2),
            -β_a_p*(x_a*x_b+p_a*p_b)/sqrt(2),
            β_b_p*x_a*p_a/sqrt(2)-κ_b_p*x_b+2*ϵ*P_x_b,
            -β_b_p*(x_a^2-p_a^2)/2/sqrt(2)-ε_d_p/sqrt(2)-κ_b_p*p_b+2*ϵ*P_p_b
            ]
    return(grad)
end

function f(u,v,params,α)
    x_a, p_a, x_b, p_b = (u+v)/2
    P_x_a, P_p_a, P_x_b, P_p_b = (u-v)/2/α
    res = α*grad_phi_H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params) - grad_theta_H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params)
    return res
end

function g(u,v,params,α)
    x_a, p_a, x_b, p_b = (u+v)/2
    P_x_a, P_p_a, P_x_b, P_p_b = (u-v)/2/α
    res = α*grad_phi_H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params) + grad_theta_H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params)
    return res
end

function H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params)
    β_a_p,β_b_p,ε_d_p,κ_b_p,ϵ = params
    
    return (β_a_p*(x_a*p_b-p_a*x_b)/sqrt(2))*P_x_a +
           (-β_a_p*(x_a*x_b+p_a*p_b)/sqrt(2))*P_p_a +
            (β_b_p*x_a*p_a/sqrt(2)-κ_b_p*x_b)*P_x_b +
            (-β_b_p*(x_a^2-p_a^2)/2/sqrt(2)-ε_d_p/sqrt(2)-κ_b_p*p_b)*P_p_b +
            ϵ*(P_x_b^2+P_p_b^2)
end

function λ_step(u::AbstractVector, v::AbstractVector, dsϕ::AbstractVector, params, α)
    x_a, x_b, p_a, p_b = (u .+ v) ./ 2
    P_x_a, P_p_a, P_x_b, P_p_b = (u .- v) ./ (2α)

    gradθ = grad_theta_H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params)
    Hval  = H(x_a,p_a,x_b,p_b,P_x_a,P_p_a,P_x_b,P_p_b,params)

    dot_term = dot(gradθ, dsϕ)
    norm2    = dot(dsϕ, dsϕ)

    disc = dot_term^2 - 4*Hval*norm2
    return (abs(dot_term) + sqrt(max(0.0, disc))) / (2*norm2)
end

function λ(u_all::AbstractMatrix, v_all::AbstractMatrix, params, α)
    Mplus1 = size(u_all, 1)
    M = Mplus1 - 1

    ϕ_all = (u_all .+ v_all) ./ 2
    λ_vals = zeros(Float64, M)              # λ_i for i=0..M-1 (Julia 1..M)

    @inbounds for i in 1:M
        dsϕ = view(ϕ_all, i+1, :) .- view(ϕ_all, i, :)
        λ_vals[i] = λ_step(view(u_all,i,:), view(v_all,i,:), dsϕ, params, α)
    end
    return λ_vals
end

In [ ]:
function update_u(u, v, params, α, ϕ_b, Δτ, Δs)
    Mplus1 = size(u, 1)
    M = Mplus1 - 1

    λ_vals = λ(u, v, params, α)

    u_new = zeros(Float64, Mplus1, 4)

    # u_M^{n+1} = -v_M^n + 2 ϕ_b
    u_new[Mplus1, :] .= ϕ_b

    # backward recursion: i = M-1,...,0  (Julia: rows M,...,1)
    @inbounds for i in M:-1:2
        num = (λ_vals[i] .* view(u_new, i+1, :) ./ Δs) .+ f(view(u,i,:), view(v,i,:), params, α)
        den = (1/Δτ) + (λ_vals[i] / Δs)
        u_new[i, :] .= num ./ den
    end

    u_new[1, :] .= ϕ_a

    return u_new, λ_vals
end

function update_v(u, v, params, α, ϕ_a, Δτ, Δs)
    Mplus1 = size(v, 1)
    M = Mplus1 - 1

    # λ_i(u^{n+1}, v^n): defined on segments i=0..M-1 (Julia 1..M)
    λ_vals = λ(u, v, params, α)

    v_new = zeros(Float64, Mplus1, 4)

    # v_0^{n+1} = -u_0^n + 2 ϕ_a   (row 1)
    v_new[1, :] .= ϕ_a

    # forward recursion: i = 1..M  (Julia: rows 2..M+1)
    @inbounds for i in 2:(M)
        num = (λ_vals[i-1] .* view(v_new, i-1, :) ./ Δs) .+ g(view(u, i, :), view(v, i, :), params, α)
        den = (1/Δτ) + (λ_vals[i-1] / Δs)
        v_new[i, :] .= num ./ den
    end

    v_new[Mplus1, :] .= ϕ_b

    return v_new, λ_vals
end

# Linear interpolation helper between rows
@inline function lerp_row(A::AbstractMatrix, k0::Int, k1::Int, t::Float64)
    return (1-t) .* view(A, k0, :) .+ t .* view(A, k1, :)
end

function reparametrize_string!(u_new::AbstractMatrix, v_new::AbstractMatrix, α;
                               n_fine::Int=10_000)
    Mplus1 = size(u_new, 1)
    M = Mplus1 - 1

    # Build φ and θ from u,v
    ϕ = (u_new .+ v_new) ./ 2
    θ = (u_new .- v_new) ./ (2α)

    # Arc-length along φ (polyline)
    s = zeros(Float64, Mplus1)
    @inbounds for i in 2:Mplus1
        d = view(ϕ, i, :) .- view(ϕ, i-1, :)
        s[i] = s[i-1] + norm(d)
    end
    L = s[end]

    # Targets: equally spaced in arc-length
    s_target = collect(range(0.0, L, length=Mplus1))

    # Interpolate φ and θ onto s_target
    ϕr = similar(ϕ)
    θr = similar(θ)

    @inbounds for i in 1:Mplus1
        st = s_target[i]
        k = searchsortedfirst(s, st)

        if k <= 1
            ϕr[i, :] .= view(ϕ, 1, :)
            θr[i, :] .= view(θ, 1, :)
        elseif k > Mplus1
            ϕr[i, :] .= view(ϕ, Mplus1, :)
            θr[i, :] .= view(θ, Mplus1, :)
        else
            s0, s1 = s[k-1], s[k]
            t = (st - s0) / (s1 - s0)

            ϕr[i, :] .= lerp_row(ϕ, k-1, k, t)
            θr[i, :] .= lerp_row(θ, k-1, k, t)
        end
    end

    # Enforce exact endpoints (recommended)
    ϕr[1, :]   .= ϕ[1, :]
    ϕr[end, :] .= ϕ[end, :]
    θr[1, :]   .= θ[1, :]
    θr[end, :] .= θ[end, :]

    # Rebuild u,v from reparametrized φ,θ
    u_new .= ϕr .+ α .* θr
    v_new .= ϕr .- α .* θr

    return nothing
end

θ_init = zeros((M+1,4))
u_init = ϕ_init + α* θ_init
v_init = ϕ_init - α* θ_init
Δs = 1/M
Δτ = 1e-3
u=u_init
v=v_init
N_iter = 100
for n in 1:N_iter
    u_new, _ = update_u(u, v, params, α, ϕ_b, Δτ, Δs)          # backward
    ϕ_inter = (u_new .+ v) ./ 2
    #println(ϕ_inter[1,:])
    #println(ϕ_inter[end,:])
    v_new, _ = update_v(u_new, v, params, α, ϕ_a, Δτ, Δs)   # forward
    ϕ_inter = (u_new .+ v_new) ./ 2
    #println(ϕ_inter[1,:])
    #println(ϕ_inter[end,:])

    reparametrize_string!(u_new, v_new, α)
    #ϕ_new = (u_new .+ v_new) ./ 2
    #gaps = [norm(view(ϕ_new,i+1,:) .- view(ϕ_new,i,:)) for i in 1:M]

    u, v = u_new, v_new
end


In [ ]:
ϕ_fin = (u+v)/2
plot(ϕ_fin[:, 1], ϕ_fin[:, 2])
xlims!(-.3,.3)
#ylims!(-.3,.3)

In [ ]:
ϕ_fin = (u+v)/2
plot(ϕ_fin[:, 3], ϕ_fin[:, 4])
xlims!(-.3,.3)
ylims!(-1.6,.3)

In [ ]:
#plot(ϕ_fin[:, 1])
plot((0:M)./(M) , ϕ_fin[:, 2], label ="Memory momentum")
#plot!(ϕ_fin[:, 3])
plot!((0:M)./(M), ϕ_fin[:, 4], label ="Buffer momentum")
xlabel!("Path parametrization")

In [ ]:
θ_fin = (u-v)/2/α
plot((0:M)./(M) , θ_fin[:, 1], label ="Conj. of Memory position")
plot!((0:M)./(M) , θ_fin[:, 2], label ="Conj. of Memory momentum")
plot!((0:M)./(M), θ_fin[:, 3], label ="Conj. of Buffer position")
plot!((0:M)./(M), θ_fin[:, 4], label ="Conj. of Buffer momentum")
xlabel!("Path parametrization")

In [ ]:
H_val = zeros(Float64,M+1)
for point in 1:M+1
    x_a, p_a, x_b, p_b = ϕ_fin[point, :]
    P_x_a, P_p_a, P_x_b, P_p_b = θ_fin[point, :]
    H_val[point] = H(x_a, p_a, x_b, p_b,P_x_a, P_p_a, P_x_b, P_p_b,params)
end
plot(H_val)